This project analyzes Billboard chart data to measure market concentration in the music industry.

We focus on:
- Data cleaning
- Artist frequency distribution
- Herfindahl-Hirschman Index (HHI)
- Market concentration trends

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect("data.db")
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,BillboardCharts
1,IndividualArtistMentions
2,SongArtistMentions
3,SongArtistChartAppearances
4,SpotifyDailyStreamingCharts
5,SpotifyWeeklyStreamingCharts
6,AppleChartRankings


In [31]:
df = pd.read_sql_query("SELECT * FROM BillboardCharts LIMIT 10;", conn)
df

,artist,song_name,rank,artist_link,last_week_rank,peak,weeks_chart,week,chart
0,Flipperachi,FA9LA,1,None,1,1,10,02-28-2026,Billboard Arabic Hot 100
1,DYSTINCT,Yama,2,None,2,1,17,02-28-2026,Billboard Arabic Hot 100
2,Vanco Featuring AYA,Ma Tnsani,3,None,3,2,29,02-28-2026,Billboard Arabic Hot 100
3,"Lege-Cy, Ghaliaa",Msh Awl Marra,4,None,7,4,3,02-28-2026,Billboard Arabic Hot 100
4,DYSTINCT,Ta3al,5,None,4,3,4,02-28-2026,Billboard Arabic Hot 100
5,Sherine Abdel Wahab,Batmanna Ansak,6,None,5,5,35,02-28-2026,Billboard Arabic Hot 100
6,TUL8TE,El Hob Gany,7,None,6,2,19,02-28-2026,Billboard Arabic Hot 100
7,"Saint Levant, Marwan Moussa",KALAMANTINA,8,None,8,2,52,02-28-2026,Billboard Arabic Hot 100
8,"Nassif Zeytoun, Abu Ward",Kazdoura,9,None,9,3,9,02-28-2026,Billboard Arabic Hot 100
9,TUL8TE,Heseeny,10,None,11,9,28,02-28-2026,Billboard Arabic Hot 100


In [3]:
df.columns

Index(['artist', 'song_name', 'rank', 'artist_link', 'last_week_rank', 'peak',
       'weeks_chart', 'week', 'chart'],
      dtype='str')


We define the market as the Billboard Arabic Hot 100 chart.
Each week consists of ranked songs, and artists compete for chart presence.

We measure market concentration using artist frequency within the chart.

In [4]:
df.shape

(5, 9)

In [5]:
df['week'].nunique()

1

In [6]:
df = pd.read_sql_query("SELECT * FROM BillboardCharts;", conn)

df.shape

(1650, 9)

In [7]:
df['week'].nunique()

2

In [8]:
sorted(df['week'].unique())[:5]

['02-21-2026', '02-28-2026']

In [9]:
df['chart'].unique()

<ArrowStringArray>
[         'Billboard Arabic Hot 100',       'Billboard Argentina Hot 100',
          'Billboard Brasil Hot 100',        'Billboard Colombia Hot 100',
        'Billboard Canadian Hot 100',           'Billboard Italy Hot 100',
           'Billboard Japan Hot 100',           'Billboard Korea Hot 100',
     'Billboard Philippines Hot 100', 'Billboard Thailand Top Thai Songs',
         'Billboard Vietnam Hot 100',                   'Australia Songs',
                     'Austria Songs',                     'Belgium Songs',
                     'Bolivia Songs',                       'Chile Songs',
               'China TME UNI Chart',                     'Croatia Songs',
              'Czech Republic Songs',                     'Denmark Songs',
                     'Ecuador Songs',                     'Finland Songs',
                      'France Songs',                     'Germany Songs',
                      'Greece Songs',                   'Hong Kong Songs',
      

Each country-specific Billboard chart is treated as an independent music market.

We calculate the Herfindahl-Hirschman Index (HHI) for each country to measure artist concentration within that market.

In [10]:
import pandas as pd
import matplotlib.pyplot as plt

df['week'] = pd.to_datetime(df['week'])

In [11]:
hhi_country = df.groupby('chart')['artist'] \
    .apply(lambda x: (x.value_counts(normalize=True) ** 2).sum())

hhi_country = hhi_country.sort_values(ascending=False)

hhi_country.head(10)

chart
Billboard Korea Hot 100    0.3952
Iceland Songs              0.2384
Hong Kong Songs            0.1616
Croatia Songs              0.1520
Ecuador Songs              0.1392
Peru Songs                 0.1360
Bolivia Songs              0.1328
Chile Songs                0.1296
New Zealand Songs          0.1168
Mexico Songs               0.1104
Name: artist, dtype: float64

## 1. Compute Country-Level Artist Market Shares

For each country chart, we compute artist market shares based on chart appearances.
Market share is defined as the proportion of songs attributed to each artist within a given country.

In [12]:
# artist market share
country_artist_share = df.groupby(['chart', 'artist']) \
    .size() \
    .reset_index(name='count')

total_songs_country = country_artist_share.groupby('chart')['count'].transform('sum')

country_artist_share['market_share'] = country_artist_share['count'] / total_songs_country

country_artist_share.head()

,chart,artist,count,market_share
0,Australia Songs,Alex Warren,1,0.04
1,Australia Songs,Bad Bunny,1,0.04
2,Australia Songs,Bruno Mars,1,0.04
3,Australia Songs,"Dave, Tems",1,0.04
4,Australia Songs,Djo,1,0.04


## 2. Herfindahl-Hirschman Index (HHI)

HHI measures market concentration as the sum of squared market shares.
Higher values indicate greater dominance by a few artists.

In [13]:
hhi_country = country_artist_share.groupby('chart')['market_share'] \
    .apply(lambda x: (x ** 2).sum()) \
    .reset_index(name='HHI')

hhi_country.head()

hhi_country.to_csv("hhi_country.csv", index=False)

## 3. Concentration Ratio (CR3)

CR3 measures the combined market share of the top 3 artists in each country.

In [14]:
cr3_country = country_artist_share.sort_values(
    ['chart', 'market_share'],
    ascending=[True, False]
).groupby('chart').head(3)

cr3_country = cr3_country.groupby('chart')['market_share'] \
    .sum() \
    .reset_index(name='CR3')

cr3_country.head()

cr3_country.to_csv("cr3_country.csv", index=False)

## 4. Gini Coefficient

The Gini coefficient measures inequality in artist market shares.
Values closer to 1 indicate greater inequality.

In [15]:
import numpy as np

def gini(array):
    array = np.sort(array)
    n = len(array)
    cumulative = np.cumsum(array)
    return 1 - 2 * np.sum(cumulative) / (n * cumulative[-1]) + 1/n

gini_country = country_artist_share.groupby('chart')['market_share'] \
    .apply(gini) \
    .reset_index(name='Gini')

gini_country.head()

gini_country.to_csv("gini_country.csv", index=False)

## 5. Summary of Concentration Measures

In [16]:
summary_table = hhi_country.merge(cr3_country, on='chart') \
    .merge(gini_country, on='chart')

summary_table = summary_table.sort_values('HHI', ascending=False)

summary_table.head(10)

summary_table.to_excel("concentration_results.xlsx", index=False)

In [17]:
song_concentration = df.groupby(['chart', 'song_name']) \
    .size() \
    .reset_index(name='appearances')

song_summary = song_concentration.groupby('chart')['appearances'] \
    .agg(['mean', 'std', 'max']) \
    .reset_index()

song_summary.head()

,chart,mean,std,max
0,Australia Songs,1.0,0.0,1
1,Austria Songs,1.0,0.0,1
2,Belgium Songs,1.0,0.0,1
3,Billboard Arabic Hot 100,1.0,0.0,1
4,Billboard Argentina Hot 100,1.0,0.0,1


In [18]:
## Top 10 Most Concentrated Markets
summary_table.sort_values('HHI', ascending=False).head(10)


,chart,HHI,CR3,Gini
10,Billboard Korea Hot 100,0.3952,0.80,0.565000
27,Iceland Songs,0.2384,0.68,0.472727
25,Hong Kong Songs,0.1616,0.60,0.410000
17,Croatia Songs,0.1520,0.56,0.387692
20,Ecuador Songs,0.1392,0.48,0.327500
37,Peru Songs,0.1360,0.48,0.322500
14,Bolivia Songs,0.1328,0.44,0.296471
15,Chile Songs,0.1296,0.40,0.264444
35,New Zealand Songs,0.1168,0.44,0.291765
33,Mexico Songs,0.1104,0.52,0.325333


In [19]:
## Top 10 Least Concentrated Markets
summary_table.sort_values('HHI').head(10)

,chart,HHI,CR3,Gini
51,Worldwide Top 200 Songs,0.01705,0.15,2.976336e-01
6,Billboard Canadian Hot 100,0.02820,0.22,2.797101e-01
50,U.S. Hot 100 Songs,0.03220,0.22,3.443333e-01
5,Billboard Brasil Hot 100,0.04000,0.12,1.873501e-16
13,Billboard Vietnam Hot 100,0.04320,0.16,3.833333e-02
32,Malaysia Songs,0.04320,0.16,3.833333e-02
28,India Songs,0.04640,0.20,7.304348e-02
45,Sweden Songs,0.04640,0.20,7.304348e-02
43,South Africa Songs,0.04640,0.20,7.304348e-02
26,Hungary Songs,0.04960,0.20,7.652174e-02


In [20]:
spotify_cols = pd.read_sql_query(
    "SELECT * FROM SpotifyWeeklyStreamingCharts LIMIT 5;",
    conn
)
apple_cols = pd.read_sql_query(
    "SELECT * FROM AppleChartRankings LIMIT 5;",
    conn
)

print("SpotifyWeeklyStreamingCharts columns:")
print(spotify_cols.columns.tolist())
print()
print("AppleChartRankings columns:")
print(apple_cols.columns.tolist())

spotify_cols.head(), apple_cols.head()

SpotifyWeeklyStreamingCharts columns:
['rank', 'artist', 'song_name', 'peak', 'weeks', 'Streams', 'Streams+', 'Total', 'chart_name', 'date', 'country']

AppleChartRankings columns:
['rank', 'artist', 'song_name', 'chart_name', 'date', 'country']


(   rank       artist          song_name  peak  weeks   Streams    Streams+  \
 0     1    Bad Bunny               DtMF     1     60  45722046 -14850248.0   
 1     2    Bad Bunny  BAILE INoLVIDABLE     2     60  33278386 -10608661.0   
 2     3          Djo   End of Beginning     1    106  32979053  -2245987.0   
 3     4    Bad Bunny           NUEVAYoL     3     60  32180613 -10955089.0   
 4     5  Olivia Dean         Man I Need     3     27  31180371  -1623349.0   
 
         Total                    chart_name        date country  
 0  1549114511  Spotify Weekly Chart Global   02-26-2026  Global  
 1  1273769628  Spotify Weekly Chart Global   02-26-2026  Global  
 2  2225140553  Spotify Weekly Chart Global   02-26-2026  Global  
 3  1040380429  Spotify Weekly Chart Global   02-26-2026  Global  
 4   792309416  Spotify Weekly Chart Global   02-26-2026  Global  ,
    rank          artist            song_name                  chart_name  \
 0     1       Bad Bunny                 DtM

In [ ]:
import numpy as np

def gini(array):
    array = np.array(array, dtype=float)
    array = np.sort(array)
    n = len(array)
    
    if n == 0:
        return np.nan
    
    cumulative = np.cumsum(array)
    
    if cumulative[-1] == 0:
        return 0.0
    
    return 1 - 2 * np.sum(cumulative) / (n * cumulative[-1]) + 1 / n


def compute_concentration_from_table(df_input, group_col='chart', entity_col='artist'):
    share_df = df_input.groupby([group_col, entity_col]) \
        .size() \
        .reset_index(name='count')

    total_count = share_df.groupby(group_col)['count'].transform('sum')
    share_df['market_share'] = share_df['count'] / total_count

    hhi_df = share_df.groupby(group_col)['market_share'] \
        .apply(lambda x: (x ** 2).sum()) \
        .reset_index(name='HHI')

    cr3_df = share_df.sort_values(
        [group_col, 'market_share'],
        ascending=[True, False]
    ).groupby(group_col).head(3)

    cr3_df = cr3_df.groupby(group_col)['market_share'] \
        .sum() \
        .reset_index(name='CR3')

    gini_df = share_df.groupby(group_col)['market_share'] \
        .apply(gini) \
        .reset_index(name='Gini')

    summary_df = hhi_df.merge(cr3_df, on=group_col) \
        .merge(gini_df, on=group_col) \
        .sort_values('HHI', ascending=False)

    return share_df, summary_df

In [22]:
spotify_df = pd.read_sql_query(
    "SELECT * FROM SpotifyWeeklyStreamingCharts;",
    conn
)

spotify_df.head()

,rank,artist,song_name,peak,weeks,Streams,Streams+,Total,chart_name,date,country
0,1,Bad Bunny,DtMF,1,60,45722046,-14850248.0,1549114511,Spotify Weekly Chart Global,02-26-2026,Global
1,2,Bad Bunny,BAILE INoLVIDABLE,2,60,33278386,-10608661.0,1273769628,Spotify Weekly Chart Global,02-26-2026,Global
2,3,Djo,End of Beginning,1,106,32979053,-2245987.0,2225140553,Spotify Weekly Chart Global,02-26-2026,Global
3,4,Bad Bunny,NUEVAYoL,3,60,32180613,-10955089.0,1040380429,Spotify Weekly Chart Global,02-26-2026,Global
4,5,Olivia Dean,Man I Need,3,27,31180371,-1623349.0,792309416,Spotify Weekly Chart Global,02-26-2026,Global


In [24]:
print(spotify_df.columns.tolist())
spotify_df.head()

['rank', 'artist', 'song_name', 'peak', 'weeks', 'Streams', 'Streams+', 'Total', 'chart_name', 'date', 'country']


,rank,artist,song_name,peak,weeks,Streams,Streams+,Total,chart_name,date,country
0,1,Bad Bunny,DtMF,1,60,45722046,-14850248.0,1549114511,Spotify Weekly Chart Global,02-26-2026,Global
1,2,Bad Bunny,BAILE INoLVIDABLE,2,60,33278386,-10608661.0,1273769628,Spotify Weekly Chart Global,02-26-2026,Global
2,3,Djo,End of Beginning,1,106,32979053,-2245987.0,2225140553,Spotify Weekly Chart Global,02-26-2026,Global
3,4,Bad Bunny,NUEVAYoL,3,60,32180613,-10955089.0,1040380429,Spotify Weekly Chart Global,02-26-2026,Global
4,5,Olivia Dean,Man I Need,3,27,31180371,-1623349.0,792309416,Spotify Weekly Chart Global,02-26-2026,Global


In [25]:
spotify_share, spotify_summary = compute_concentration_from_table(
    spotify_df,
    group_col='chart_name',
    entity_col='artist'
)

spotify_summary.head(10)

,chart_name,HHI,CR3,Gini
19,Spotify Weekly Chart El Salvador,0.04775,0.310,0.510260
34,Spotify Weekly Chart Israel,0.04400,0.270,0.497561
75,Spotify Weekly Chart Venezuela,0.04385,0.310,0.511528
27,Spotify Weekly Chart Honduras,0.04010,0.275,0.493580
16,Spotify Weekly Chart Dominican Republic,0.03895,0.275,0.505897
43,Spotify Weekly Chart Mexico,0.03600,0.250,0.479024
12,Spotify Weekly Chart Costa Rica,0.03565,0.255,0.448261
36,Spotify Weekly Chart Japan,0.03520,0.235,0.496757
47,Spotify Weekly Chart Nicaragua,0.03520,0.260,0.462299
53,Spotify Weekly Chart Peru,0.03365,0.230,0.416667


In [26]:
spotify_summary.to_csv("spotify_weekly_concentration_measures.csv", index=False)
spotify_share.to_csv("spotify_weekly_artist_shares.csv", index=False)

In [27]:
apple_df = pd.read_sql_query(
    "SELECT * FROM AppleChartRankings;",
    conn
)

print(apple_df.columns.tolist())
apple_df.head()

['rank', 'artist', 'song_name', 'chart_name', 'date', 'country']


,rank,artist,song_name,chart_name,date,country
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026,Worldwide
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026,Worldwide
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026,Worldwide
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026,Worldwide
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026,Worldwide


In [28]:
apple_df = pd.read_sql_query(
    "SELECT * FROM AppleChartRankings;",
    conn
)

apple_df.head()

,rank,artist,song_name,chart_name,date,country
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026,Worldwide
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026,Worldwide
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026,Worldwide
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026,Worldwide
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026,Worldwide


In [29]:
apple_share, apple_summary = compute_concentration_from_table(
    apple_df,
    group_col='chart_name',
    entity_col='artist'
)

apple_summary.head(10)

,chart_name,HHI,CR3,Gini
37,CÃ´te d'Ivoire Apple Music Top Song,0.087850,0.4025,0.600479
31,China Apple Music Top Song,0.078575,0.3750,0.556169
63,Israel Apple Music Top Song,0.062350,0.3275,0.586754
14,Benin Apple Music Top Song,0.058687,0.3500,0.576703
23,Burkina Faso Apple Music Top Song,0.052550,0.3200,0.399851
59,Iceland Apple Music Top Song,0.049325,0.2625,0.628041
118,Senegal Apple Music Top Song,0.048562,0.2825,0.497250
92,Montserrat Apple Music Top Song,0.048200,0.2900,0.431359
66,Japan Apple Music Top Song,0.047738,0.3075,0.559627
120,Seychelles Apple Music Top Song,0.045163,0.3050,0.555341


In [30]:
apple_summary.to_csv("apple_chart_concentration_measures.csv", index=False)
apple_share.to_csv("apple_chart_artist_shares.csv", index=False)